In [ ]:
#Librerias
import pandas as pd
import matplotlib.pyplot as plt

#Funciones
def dataframe():
    path = "/content/drive/MyDrive/Proyecto #1/dataset_final/"
    dataset = "Dataset_final.csv"
    df = pd.read_csv(path + dataset)
    return df

def variable_fecha_reciente(df):
    cantidad_de_filas = df["Fecha"].count() - 1
    ultima_fecha = df.loc[cantidad_de_filas, "Fecha"]
    ultima_fecha = str(ultima_fecha)
    ultima_fecha = ultima_fecha[:10]
    df.set_index("Fecha", inplace=True)
    return df, ultima_fecha, cantidad_de_filas

def variables_calculos_diferencias_salario(df, cantidad_de_filas):
    salario = df.loc[:,"Salario - % acumulado"].iloc[cantidad_de_filas]
    inflacion = df.loc[:,"Inflación acumulada (%)"].iloc[cantidad_de_filas]
    dolar = df.loc[:, "Dólar - % acumulado"].iloc[cantidad_de_filas]
    #Calculamos la diferencia
    diferencia_salario_x_inflacion = (inflacion - salario) / salario * 100
    diferencia_salario_x_dolar = (dolar - salario) / salario * 100
    #Redondeamos los decimales
    diferencia_salario_x_inflacion = diferencia_salario_x_inflacion.round(2)
    diferencia_salario_x_dolar = diferencia_salario_x_dolar.round(2)
    return diferencia_salario_x_dolar, diferencia_salario_x_inflacion, salario, inflacion, dolar

def variable_calculo_diferencia_dolar(inflacion, dolar):
    diferencia_dolar = (inflacion - dolar) / dolar * 100
    diferencia_dolar = diferencia_dolar.round(2)
    return diferencia_dolar

# Esta función recorre el DataFrame para identificar el valor máximo en todo el período y seleccionar todos los meses del año de dicho periodo.
# Devuelve un DataFrame filtrado, listo para ser utilizado en graficos.
def variables_graficos(df, ultima_fecha):
    lista = ["Salario - Variación mensual (%)", "Dólar - Variación mensual (%)", "Inflación mensual (%)"]
    lista_numero = 0
    ultima_fecha_año = int(ultima_fecha[:4])
    ultima_fecha_mes_y_dia = ultima_fecha[5:]
    while lista_numero <= 2:
       columna = lista[lista_numero]
       numero = df[columna].idxmax()
       numero = int(numero[:4])
       if numero < ultima_fecha_año:
          numero = str(numero)
          fecha_1 = numero+"-01-01"
          fecha_2 = numero+"-12-01"
       if numero == ultima_fecha_año:
          numero -= 1
          numero = str(numero)
          fecha_1 = numero+ultima_fecha_mes_y_dia
          fecha_2 = ultima_fecha.copy()
       else:
           if lista_numero == 0:
              grafico_salario = df.loc[fecha_1:fecha_2].loc[:, columna]
           if lista_numero == 1:
              grafico_dolar = df.loc[fecha_1:fecha_2].loc[:, columna]
           if lista_numero == 2:
              grafico_inflacion = df.loc[fecha_1:fecha_2].loc[:, columna]
       lista_numero += 1
    return grafico_dolar, grafico_salario, grafico_inflacion

def salario_minimo_grafico(df, grafico_salario):
   print("Salario Minimo:")
   print("* El valor promedio mensual es de", df["Salario - Variación mensual (%)"].mean().round(2), "%.")
   print("* El valor mas alto en la variacion mensual es de", df["Salario - Variación mensual (%)"].max(), "%, que fue en la fecha", df["Salario - Variación mensual (%)"].idxmax()+".")
   print("* El valor más bajo en la variación mensual es de", df["Salario - Variación mensual (%)"].min(), "%, y se repite en varias fechas. Sí, hubo muchos meses en los que el salario no tuvo ningún aumento.")
   print("")
   grafico_salario.plot(kind='bar', title= "Salario - Variación mensual (%)", color="black", xlabel='')
   plt.show()
   return

def dolar_grafico(df, grafico_dolar):
    print("Dólar Oficial:")
    print("* El valor promedio mensual es de", df["Dólar - Variación mensual (%)"].mean().round(2), "%.")
    print("* El valor mas alto en la variacion mensual es de", df["Dólar - Variación mensual (%)"].max(), "%, que fue en la fecha", df["Dólar - Variación mensual (%)"].idxmax()+".")
    print("* El valor mas bajo en la variacion mensual es de", df["Dólar - Variación mensual (%)"].min(), "%, que fue en la fecha", df["Dólar - Variación mensual (%)"].idxmin()+".")
    print("")
    grafico_dolar.plot(kind='bar', title= "Dólar - Variación mensual (%)", color="green", xlabel='')
    plt.show()
    return

def inflacion_grafico(df, grafico_inflacion):
    print("Inflación:")
    print("* El valor promedio mensual es de", df["Inflación mensual (%)"].mean().round(2), "%.")
    print("* El valor mas alto en la variacion mensual es de", df["Inflación mensual (%)"].max(), "%, que fue en la fecha", df["Inflación mensual (%)"].idxmax()+".")
    print("* El valor mas bajo en la variacion mensual es de", df["Inflación mensual (%)"].min(), "%, que fue en la fecha", df["Inflación mensual (%)"].idxmin()+".")
    print("")
    grafico_inflacion.plot(kind='bar', title= "Inflación mensual (%)", color="red", xlabel='')
    plt.show()
    return

def grafico_acumulado(df, ultima_fecha, salario, dolar, inflacion):
    df[["Dólar - % acumulado", "Inflación acumulada (%)", "Salario - % acumulado"]].plot(kind='line',title= "Acumulado (%)", xlabel='', color=["green", "red", "black"])
    plt.show()
    print("Ultima actualizacion:", ultima_fecha)
    print("* Salario acumulado:", salario, "%")
    print("* Inflacion acumulada:", inflacion, "%")
    print("* Dolar acumulado:", dolar, "%")
    return

def conclusion(diferencia_salario_x_inflacion, diferencia_salario_x_dolar, ultima_fecha, diferencia_dolar):
    print("\nEl salario mínimo creció un",diferencia_salario_x_inflacion,"% menos que la inflación y un",diferencia_salario_x_dolar ,"% menos que el dólar oficial en el mismo período (2017-01-01 a",ultima_fecha+").")
    print("Esto demuestra la gran pérdida de poder adquisitivo frente a ambos indicadores, con una diferencia abismal.")
    print("Si observamos el gráfico anterior de acumulación, el salario mínimo nunca repuntó ni estuvo cerca de la inflación ni del dólar.")
    print("¿Qué significa esto? Se podría interpretar de diferentes maneras:")
    print("")
    print('* "Los gobiernos priorizaron otros factores macroeconómicos (guiño) menos el más importante."')
    print('* "Todos saben, todos son expertos en economía, pero cuando toman el puesto, demuestran lo contrario."')
    print('* "La culpa es de otro."')
    print("")
    print("También se puede observar que hay una diferencia del", diferencia_dolar,"% entre el dólar oficial y la inflación.")
    print("Generalmente, el dólar suele acompañar el movimiento de los precios, por lo que esta brecha podría indicar que el tipo de cambio oficial se encuentra atrasado respecto al ritmo inflacionario.")
    print("En resumen, con el salario mínimo actual, tenés mucho menos poder de compra, tanto local como internacionalmente, que a principios de 2017.")
    return





